In [ ]:
import pandas as pd
import numpy as np
import warnings
import tensorflow as tf

from i import input_dir, output_dir, model_dir
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

warnings.filterwarnings("ignore")

keras = tf.keras
layers = keras.layers
EarlyStopping = keras.callbacks.EarlyStopping

train = pd.read_csv(input_dir + "train.csv", index_col="id")
test = pd.read_csv(input_dir + "test.csv", index_col="id")


for col in train.select_dtypes(include="object").columns:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

Target_Col = "exam_score"

X = train.iloc[:, :-1]
y = train[Target_Col]
X_test = test

In [2]:
categorical_cols = X.select_dtypes(include="category").columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

ohe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[categorical_cols] = ohe.fit_transform(X[categorical_cols])
X_test[categorical_cols] = ohe.transform(X_test[categorical_cols])
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])
y = y.values.reshape(-1, 1)
scaler_y = StandardScaler()
y = scaler_y.fit_transform(y)
X = X.values
X_test = X_test.values
kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(X_test.shape[0])
for train_index, val_index in kf.split(X):
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    model = keras.Sequential(
        [
            keras.Input(shape=(X.shape[1],)),
            layers.Dropout(0.2),
            layers.BatchNormalization(),
            layers.Dense(1024, activation="relu"),
            layers.Dropout(0.2),
            layers.BatchNormalization(),
            layers.Dense(256, activation="relu"),
            layers.Dropout(0.2),
            layers.BatchNormalization(),
            layers.Dense(1),
        ]
    )

    model.compile(optimizer="adam", loss="mse")

    early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

    model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32, callbacks=[early_stopping], verbose=0)

    val_preds = model.predict(X_val)
    val_rmse = np.sqrt(root_mean_squared_error(scaler_y.inverse_transform(y_val), scaler_y.inverse_transform(val_preds)))
    print(f"Validation RMSE: {val_rmse}")

    test_fold_preds = model.predict(X_test)
    test_preds += test_fold_preds.flatten() / kf.n_splits
test_preds = scaler_y.inverse_transform(test_preds.reshape(-1, 1)).flatten()



3938/3938 ━━━━━━━━━━━━━━━━━━━━ 2s 585us/step
Validation RMSE: 3.1104209366093416
8438/8438 ━━━━━━━━━━━━━━━━━━━━ 5s 573us/step
3938/3938 ━━━━━━━━━━━━━━━━━━━━ 2s 586us/step
Validation RMSE: 3.1056768475301513
8438/8438 ━━━━━━━━━━━━━━━━━━━━ 5s 574us/step
3938/3938 ━━━━━━━━━━━━━━━━━━━━ 2s 605us/step
Validation RMSE: 3.0936389119204484
8438/8438 ━━━━━━━━━━━━━━━━━━━━ 5s 591us/step
3938/3938 ━━━━━━━━━━━━━━━━━━━━ 2s 618us/step
Validation RMSE: 3.091176286770277
8438/8438 ━━━━━━━━━━━━━━━━━━━━ 5s 601us/step
3938/3938 ━━━━━━━━━━━━━━━━━━━━ 2s 606us/step
Validation RMSE: 3.090067071931095
8438/8438 ━━━━━━━━━━━━━━━━━━━━ 5s 595us/step


In [5]:
submission = pd.DataFrame(data= test_preds, index=X_test.index, columns=[Target_Col])
# submission[Target_Col] = test_preds
submission.to_csv(output_dir + "nn_submission.csv")